In [4]:
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'pandas'

In [ ]:
np.random.seed(42)

n = 100000   # total rows

df = pd.DataFrame({

    # Trips per year (Poisson distribution → realistic user behavior)
    "trip_frequency": np.random.poisson(lam=3, size=n),

    # Avg booking value in ₹
    "average_booking_value": np.random.randint(
        5000, 150000, size=n
    ),

    # Unique destinations visited
    "destination_diversity": np.random.randint(
        1, 10, size=n
    ),

    # Session duration in minutes
    "session_duration": np.random.randint(
        1, 120, size=n
    ),

    # Search engagement category
    "search_behavior": np.random.choice(
        ["Low", "Medium", "High"],
        size=n,
        p=[0.3, 0.5, 0.2]
    )
})

# Save clean dataset
df.to_csv("travel_user_behavior_clean_100k.csv", index=False)

print("✅ Clean dataset generated")


# ---------------------------------
# STEP 2 — Create Corrupted Version
# ---------------------------------

df_corrupt = df.copy()

# ---- 2.1 Missing Values (10%) ----
for col in df_corrupt.columns:
    df_corrupt.loc[
        df_corrupt.sample(frac=0.10).index, col
    ] = np.nan


# ---- 2.2 Duplicate Rows ----
duplicates = df_corrupt.sample(5000)
df_corrupt = pd.concat(
    [df_corrupt, duplicates],
    ignore_index=True
)


# ---- 2.3 Outliers Injection ----

# Extreme booking values
df_corrupt.loc[
    df_corrupt.sample(frac=0.05).index,
    "average_booking_value"
] *= 6

# Extreme session duration
df_corrupt.loc[
    df_corrupt.sample(frac=0.05).index,
    "session_duration"
] *= 4


# ---- 2.4 Noise in Numerical Data ----

noise_trip = np.random.normal(0, 2, len(df_corrupt))
df_corrupt["trip_frequency"] = (
    df_corrupt["trip_frequency"] + noise_trip
)

noise_session = np.random.normal(0, 5, len(df_corrupt))
df_corrupt["session_duration"] = (
    df_corrupt["session_duration"] + noise_session
)


# ---- 2.5 Categorical Corruption ----

df_corrupt["search_behavior"] = df_corrupt[
    "search_behavior"
].replace({
    "Low": "low",
    "Medium": "Med",
    "High": "HIGH"
})


# ---- 2.6 Invalid Values Injection ----

df_corrupt.loc[
    df_corrupt.sample(frac=0.02).index,
    "trip_frequency"
] = -5   # invalid negative trips


# Save corrupted dataset
df_corrupt.to_csv(
    "travel_user_behavior_corrupted_100k.csv",
    index=False
)

print("💣 Corrupted dataset generated")
